# 04 - Allocation backtest

Compare a regime-dependent risk-budget policy against robust baselines. The focus is drawdown and tail-risk reduction, not alpha marketing.


In [ ]:
from regime_portfolio.allocation import make_regime_weight_function
from regime_portfolio.backtest import run_backtest
from regime_portfolio.data import DataConfig, compute_returns, load_price_panel
from regime_portfolio.features import make_feature_panel
from regime_portfolio.metrics import compare_strategies
from regime_portfolio.regimes import RegimeDetector

prices = load_price_panel(DataConfig(start="2010-01-01", source="stooq"))
returns = compute_returns(prices)
features = make_feature_panel(prices, returns)
regimes = RegimeDetector(n_regimes=3, random_state=42).fit(features).predict(features)

strategies = {
    "equal_weight": make_regime_weight_function(method="equal_weight", budgets={0: 1.0, 1: 1.0, 2: 1.0}),
    "inverse_vol_regime": make_regime_weight_function(method="inverse_volatility", defensive_asset="SHY"),
    "minvar_regime": make_regime_weight_function(method="minimum_variance", defensive_asset="SHY"),
}

results = {}
for name, fn in strategies.items():
    results[name] = run_backtest(returns, fn, regimes=regimes, lookback=252).returns

strategy_returns = __import__("pandas").concat(results, axis=1)
print(compare_strategies(strategy_returns).round(4))
